# Experiment log file

In [ ]:
from datetime import datetime
from pathlib import Path
import pandas as pd
import json

Set path to log file name

In [ ]:
LOG_PATH = Path("../data/20181011_182540.log")

Extract information from log file, resulting in a list of stimulus presentations (`stims`, list of dictionaries) will timing information and stimulus parameters

In [ ]:
nLinesTotal = 0
nLinesData = 0
nLinesErr = 0
nErr = 0
isStimStarted = False
stims = []
nStims = 0

with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as fLog:
    for line in fLog:
        # Extract elements of each line
        nLinesTotal += 1
        sDateTime = line[:15]
        sInfoType = line[15:23].strip().upper()
        sMsg = line[23:len(line) -1].strip()

        # Convert time stamp into a datetime        
        dt = datetime.strptime(sDateTime, "%Y%m%d_%H%M%S")
        if nLinesTotal == 1:
            # First line; take as start time
            dt_log_start = dt
            dt_last_end = dt_log_start

        # Filter for relevant information
        if sInfoType not in ["DATA"]:
            # Ignore filed
            continue

        # Convert data line into dictionary 
        sMsg = sMsg.replace("'", "\"")
        sMsg = sMsg.replace("\\\\", "/")
        sMsg = sMsg.replace("(", "[")
        sMsg = sMsg.replace(")", "]")

        try:
            data = json.loads(sMsg)
        except json.JSONDecodeError as e:
            # JSON parsing failed
            print(f"ERROR: parsing line {nLinesTotal-1} failed:")
            print(f"'{sMsg}'")
            nLinesErr += 1
            data = None

        # Get stimulus start/stop pairs
        try:
            stimState = data["stimState"].upper()
        except KeyError:
            stimState = None        


        if stimState:
            # Data contains stimulus information
            if stimState == "STARTED":
                if isStimStarted:
                    print("ERROR: Two consecutive stimulus starts")
                    nErr += 1

                isStimStarted = True
                iLineLastStart = nLinesData
                dt_start = dt
                t_diff = (dt -dt_log_start).total_seconds()
                t_diff_last = (dt -dt_last_end).total_seconds()
                stimInfo = dict(
                    {"index": nStims, 
                     "stimFileName": data["stimFileName"],
                     "stimMD5": data["stimMD5"],
                     "t_abs_s": t_diff,
                     "t_since_last_s": t_diff_last,
                     "t_start": dt.time()}

                )

            elif stimState in ["ABORTED", "FINISHED"]:
                if isStimStarted:
                    # Check if stimulus end belongs to stimulus start
                    if not data["stimFileName"] == stimInfo["stimFileName"]:
                        print("ERROR: File paths for stimulus start and end differ")
                        nErr += 1

                    # Append stimulus list entry
                    dt_last_end = dt
                    t_diff = (dt -dt_start).total_seconds()
                    stimInfo.update(
                        {"aborted": stimState == "ABORTED",
                         "t_end": dt.time(),
                         "t_dur_s": t_diff}
                    )
                    stims.append(stimInfo)
                    nStims += 1
                    isStimStarted = False
                else:
                    print("ERROR: Stimulus end w/o start?")    
                    nErr += 1
        else:
            # Other information
            try:
                _ = data["nFrames"]
                isFrameInfo = True
            except KeyError:
                isFrameInfo = False

            if isFrameInfo:
                # Information about stimulus presentation statistics
                stims[nStims -1].update(
                    {"t_dur_s_calc": data["nFrames"] /data["avgFreq_Hz"],
                     "nDroppedFrames": data["nDroppedFrames"]}
                )
            else:            
                if nLinesData > iLineLastStart and nLinesData < iLineLastStart +3:
                    # Last start was only up to 2 lines before
                    stims[nStims -1].update(
                        {"params": data}
                    )
                else:
                    print("ERROR: Data w/o start??")    
                    nErr += 1

        '''
        print(f"line #{nLinesData}:")
        print(dt)
        print(data)
        '''
        nLinesData += 1

    print(f"{nLinesData} of {nLinesTotal} line(s) extracted.")
    print(f"{nLinesErr} line(s) failed parsing, {nErr} error(s) occurred post-processing.")    

280 of 1401 line(s) extracted.
0 line(s) failed parsing, 0 error(s) occurred post-processing.

  #    start   t [s] gap [s] dur [s] nFr/frq abort stimulus name
--- -------- ------- ------- ------- ------- ----- 
  0 12:57:03       3       3       0       0        `__autorun_default_DO_NOT_DELETE`
  1 13:19:26    1346    1343     189     189   y    `DN`
  2 13:22:39    1539       4      12      12   y    `DS`
  3 13:22:54    1554       3     169     169   y    `DN`
  4 13:25:44    1724       1      31      30   y    `DS`
  5 13:26:16    1756       1     147     146   y    `DN`
  6 13:28:44    1904       1     100     100   y    `DS`
  7 13:31:24    2064      60     101     101        `DS`
  8 13:33:42    2202      37     165     165        `Chirp`
  9 13:37:07    2407      40     618     618        `MouseCam_Right`
 10 13:48:54    3114      89     101     101        `DS`
 11 13:51:55    3295      80     101     101        `DS`
 12 13:54:07    3427      31     165     165        `Chirp`


Print stimulus presentation table

In [ ]:
print("  #    start   t [s] gap [s] dur [s] nFr/frq abort stimulus name")
print("--- -------- ------- ------- ------- ------- ----- ")

for stim in stims:
    s = f"{stim["index"]:3d} {stim["t_start"]} {stim["t_abs_s"]:7.0f} "
    s += f"{stim["t_since_last_s"]:7.0f} {stim["t_dur_s"]:7.0f} {stim["t_dur_s_calc"]:7.0f} "  
    s += f"{"  y   " if stim["aborted"] else "      "} "
    s += f"`{Path(stim["stimFileName"]).name}`"

    print(s)

Example for information contained in one stimulus presentation entry

In [98]:
stims[1]

{'index': 1,
 'stimFileName': 'C:/Users/AGEuler/Documents/QDSpy/Stimuli/Katrin/RGCs/DN',
 'stimMD5': '521d77a52eff75053da309ada99d1156',
 't_abs_s': 1346.0,
 't_since_last_s': 1343.0,
 't_start': datetime.time(13, 19, 26),
 'aborted': True,
 't_end': datetime.time(13, 22, 35),
 't_dur_s': 189.0,
 't_dur_s_calc': 189.06897092557847,
 'nDroppedFrames': 1,
 'params': {'barDx_um': 1000.0,
  'nFrPerMarker': 3,
  'nTrials': 3,
  'barDy_um': 300.0,
  'vel_umSec': 1000.0,
  'DirList': [0, 180, 45, 225, 90, 270, 135, 315],
  'barColor': [255, 255, 255],
  'durFr_s': 0.016666666666666666,
  'tMoveDur_s': 4.0,
  'stimFileName': 'C:/Users/AGEuler/Documents/QDSpy/Stimuli/Katrin/RGCs/DS',
  'bkgColor': [0, 0, 0]}}

In [51]:
p.name


'__autorun_default_DO_NOT_DELETE'